In [ ]:
# Download data for the whole California
import requests
from bs4 import BeautifulSoup
import os
from urllib.parse import urljoin, urlparse, parse_qs

# Base URL of the FEMA website
base_url = "https://hazards.fema.gov/femaportal/NFHL/"

# Target page URL
url = "https://hazards.fema.gov/femaportal/NFHL/searchResult"  # Update if needed

# Folder to save the downloads
download_folder = "FEMA_NFHL"
os.makedirs(download_folder, exist_ok=True)

# Get the webpage content
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

# Find all download links
download_links = []
for img_tag in soup.find_all("img"):
    parent_tag = img_tag.find_parent("a")
    if parent_tag and "href" in parent_tag.attrs:
        file_url = parent_tag["href"]
        
        # Ensure the URL is absolute
        full_url = urljoin(base_url, file_url)
        
        # Parse URL to check state parameter
        parsed_url = urlparse(full_url)
        query_params = parse_qs(parsed_url.query)

        # Filter only California-related files
        if "state" in query_params and "CALIFORNIA" in query_params["state"]:
            download_links.append(full_url)

# Download each file
for file_url in download_links:
    parsed_url = urlparse(file_url)
    query_params = parse_qs(parsed_url.query)

    # Extract filename from query parameters
    if "fileName" in query_params:
        file_name = query_params["fileName"][0]
    else:
        file_name = os.path.basename(parsed_url.path)  # Fallback
    
    file_path = os.path.join(download_folder, file_name)

    print(f"Downloading {file_url}...")
    file_response = requests.get(file_url, stream=True)

    with open(file_path, "wb") as file:
        for chunk in file_response.iter_content(chunk_size=1024):
            if chunk:
                file.write(chunk)

    print(f"Saved to {file_path}")

print("All California files downloaded successfully.")

In [ ]:
# Expanded on the above code to download data for the whole US!

import requests
from bs4 import BeautifulSoup
import os
from urllib.parse import urljoin, urlparse, parse_qs
import logging

# Base URL of the FEMA website and target search URL
base_url = "https://hazards.fema.gov/femaportal/NFHL/"
url = "https://hazards.fema.gov/femaportal/NFHL/searchResult"  # Update if needed

# Base folder to save all downloads
base_download_folder = "FEMA_NFHL"
os.makedirs(base_download_folder, exist_ok=True)

# Setup logging to a file in the base download folder
log_file = os.path.join(base_download_folder, "download_log.txt")
logging.basicConfig(filename=log_file,
                    level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')
logging.info("Started downloading FEMA NFHL files.")

# Fetch the webpage content
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

# Find all download links and associate them with their state
download_links = []
for img_tag in soup.find_all("img"):
    parent_tag = img_tag.find_parent("a")
    if parent_tag and "href" in parent_tag.attrs:
        file_url = parent_tag["href"]
        full_url = urljoin(base_url, file_url)
        parsed_url = urlparse(full_url)
        query_params = parse_qs(parsed_url.query)
        
        # Check for a state parameter in the URL query
        if "state" in query_params:
            state = query_params["state"][0]
            download_links.append((full_url, state))

# Group download links by state
state_download_links = {}
for file_url, state in download_links:
    state_download_links.setdefault(state, []).append(file_url)

# Download each file for each state into its state-specific folder
for state, urls in state_download_links.items():
    state_folder = os.path.join(base_download_folder, state)
    os.makedirs(state_folder, exist_ok=True)
    logging.info(f"Started downloading files for state: {state}")
    
    for file_url in urls:
        # Extract the file name from query parameters or fallback to basename
        parsed_url = urlparse(file_url)
        query_params = parse_qs(parsed_url.query)
        if "fileName" in query_params:
            file_name = query_params["fileName"][0]
        else:
            file_name = os.path.basename(parsed_url.path)
        file_path = os.path.join(state_folder, file_name)
        
        print(f"Downloading {file_url} into folder '{state_folder}'...")
        file_response = requests.get(file_url, stream=True)
        with open(file_path, "wb") as file:
            for chunk in file_response.iter_content(chunk_size=1024):
                if chunk:
                    file.write(chunk)
        print(f"Saved to {file_path}")
    
    # Log the completion of downloads for the current state
    logging.info(f"Finished downloading files for state: {state}. Moving to next state.")

logging.info("All files for all states downloaded successfully.")
print("All files for all states downloaded successfully. Check the log file for details.")
